### 1. Retrieve Rental Data for a Specific Month and Year

In [4]:
import mysql.connector
import pandas as pd

# Establish the connection
connection = mysql.connector.connect(
    host='localhost',
    user='root',
    password='234678',
    database='sakila'
)

# Retrieve column names from the rental table
cursor = connection.cursor()
cursor.execute("SHOW COLUMNS FROM rental")
columns = cursor.fetchall()
print("Columns in rental table:", [column[0] for column in columns])

# Define the function to retrieve rental data for a specific month and year
def rentals_month(engine, month, year):
    query = f"""
        SELECT customer_id, rental_date, inventory_id
        FROM rental
        WHERE MONTH(rental_date) = {month} AND YEAR(rental_date) = {year};
    """
    return pd.read_sql(query, engine)

# Retrieve data for May 2005
df_may_2005 = rentals_month(connection, 5, 2005)
print(df_may_2005.head())

# Close the connection
connection.close()


Columns in rental table: ['rental_id', 'rental_date', 'inventory_id', 'customer_id', 'return_date', 'staff_id', 'last_update']
   customer_id         rental_date  inventory_id
0          130 2005-05-24 22:53:30           367
1          459 2005-05-24 22:54:33          1525
2          408 2005-05-24 23:03:39          1711
3          333 2005-05-24 23:04:41          2452
4          222 2005-05-24 23:05:21          2079


C:\Users\cleid\AppData\Local\Temp\ipykernel_23888\883488921.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, engine)


### 2. Count the Number of Rentals for Each Customer

In [5]:
def rental_count_month(df, month, year):
    column_name = f'rentals_{month}_{year}'
    rental_counts = df.groupby('customer_id').size().reset_index(name=column_name)
    return rental_counts

# Count the rentals for May 2005
rental_counts_may_2005 = rental_count_month(df_may_2005, 5, 2005)
print(rental_counts_may_2005.head())


   customer_id  rentals_5_2005
0            1               2
1            2               1
2            3               2
3            5               3
4            6               3


### 3. Compare Rentals Between Two Months

In [7]:
def compare_rentals(df1, df2):
    df_combined = pd.merge(df1, df2, on='customer_id', how='outer')
    df_combined['difference'] = df_combined.iloc[:, 1] - df_combined.iloc[:, 2]
    return df_combined

# Re-establish the connection
connection = mysql.connector.connect(
    host='localhost',
    user='root',
    password='234678',
    database='sakila'
)

# Retrieve rental data for June 2005 and compare with May 2005
df_june_2005 = rentals_month(connection, 6, 2005)
rental_counts_june_2005 = rental_count_month(df_june_2005, 6, 2005)

# Compare rentals between May and June
comparison = compare_rentals(rental_counts_may_2005, rental_counts_june_2005)
print(comparison.head())

# Close the connection
connection.close()


   customer_id  rentals_5_2005  rentals_6_2005  difference
0            1             2.0             7.0        -5.0
1            2             1.0             1.0         0.0
2            3             2.0             4.0        -2.0
3            4             NaN             6.0         NaN
4            5             3.0             5.0        -2.0


C:\Users\cleid\AppData\Local\Temp\ipykernel_23888\883488921.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, engine)
